# RNA-KG Visualization
Embed node2vec skipgram -> visualiza t-SNE (p=q=1, bfs like (0.2,5) e dfs like (5,0.2))

Embed Line -> visualiza t-SNE

In [ ]:
import os
# change working directory so that it picks up the grapehelper library
os.chdir('/home/ftorgano/rna-kg-analysis')
from helper_lib import graph
from helper_lib import cache
from helper_lib import predict
from helper_lib import visualize
import importlib
importlib.reload(graph)
importlib.reload(cache)
importlib.reload(predict)
importlib.reload(visualize)
cache.set_embedding_cache_dir("./RNA-KG_notebooks/Default_RNA-KG/cache/embeddings/")
import logging
logging.basicConfig(level=logging.INFO)
logging.getLogger().setLevel(logging.INFO)

In [ ]:
from sklearn.tree import DecisionTreeClassifier 
model = DecisionTreeClassifier(max_depth=5)

save_folder='./RNA-KG_notebooks/Default_RNA-KG/reports/report2fixed/'
nodes_visualization_subsampling = 20_000
edges_visualization_subsampling = 10_000

In [ ]:
import pandas as pd
from itables import init_notebook_mode,show
init_notebook_mode(all_interactive=False)
import time
import numpy as np
import matplotlib.pyplot as plt
cycler_colors = ["#3f90da", "#ffa90e", "#bd1f01", "#94a4a2", "#832db6", "#a96b59", "#e76300", "#b9ac70", "#717581", "#92dadd"]
from cycler import cycler
plt.rcParams['axes.prop_cycle'] = cycler(color=cycler_colors)
df_formatters = {'balanced_acc_mean':'{:.2%}'.format,'balanced_acc_std':'{:.2%}'.format}

In [ ]:
undirected_RNAKG = graph.load_rnakg_fixed(directed=False)
print(undirected_RNAKG.get_number_of_nodes())
print(undirected_RNAKG.get_number_of_edges())

In [ ]:
directed_RNAKG = graph.load_rnakg_fixed(directed=True)
print(directed_RNAKG.get_number_of_nodes())
print(directed_RNAKG.get_number_of_edges())

In [ ]:
node_types_undirected = graph.get_node_types_grape(undirected_RNAKG)
node_types_directed = graph.get_node_types_grape(directed_RNAKG)
edge_types_undirected = graph.get_edge_types_grape(undirected_RNAKG)
edge_types_directed = graph.get_edge_types_grape(directed_RNAKG)

In [ ]:
num_node_types_undirected = len(np.unique(node_types_undirected))
num_node_types_directed = len(np.unique(node_types_directed))
num_edge_types_undirected = len(np.unique(edge_types_undirected))
num_edge_types_directed = len(np.unique(edge_types_directed))

In [ ]:
df_node_type_pred = pd.DataFrame(columns=['embedding_name','is_directed','num_classes','balanced_acc_mean','balanced_acc_std'])
df_edge_type_pred = pd.DataFrame(columns=['embedding_name','is_directed','num_classes','balanced_acc_mean','balanced_acc_std'])

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import ShuffleSplit, StratifiedShuffleSplit
import numpy as np
from collections import Counter

def calculate_balanced_accuracy_from_vis(visualizer, random_state=42, keep_top_classes=0):
  nodes_embedded = visualizer._node_decomposition
  node_types = visualizer._get_flatten_multi_label_and_unknown_node_types()
  # print(type(nodes_embedded))
  # print(nodes_embedded)
  # print(type(node_types))
  # print(node_types)
  return calculate_balanced_accuracy(nodes_embedded, node_types, visualizer._graph, random_state, keep_top_classes)

def calculate_balanced_accuracy(nodes_embedded, node_types, graph, random_state=42, keep_top_classes=0):
  # Sets the top k most frequent node types to be the only ones, the rest are set to k
  if(keep_top_classes>0):
    counts = np.bincount(node_types)
    node_type_names_iter = (
      graph.get_node_type_name_from_node_type_id(node_id)
      for node_id in range(graph.get_number_of_node_types())
    )
    node_type_names = np.array(
      list(node_type_names_iter),
      dtype=str,
    )
    top_counts = [
      index
      for index, _ in sorted(
          enumerate(zip(counts, node_type_names)), key=lambda x: x[1], reverse=True
      )[:keep_top_classes]
    ]
    #print(top_counts)
    for i, element_type in enumerate(node_types):
      if element_type not in top_counts:
        node_types[i] = keep_top_classes
      else:
        node_types[i] = top_counts.index(element_type)

  if min(Counter(node_types).values())==1:
    SplitterClass = ShuffleSplit
  else: 
    SplitterClass = StratifiedShuffleSplit

  test_accuracies = []

  for train_indices, test_indices in SplitterClass(
    n_splits=5,
    test_size=0.3,
    random_state=random_state
  ).split(nodes_embedded, node_types):
    model = DecisionTreeClassifier(max_depth=5)

    train_x, test_x = nodes_embedded[train_indices], nodes_embedded[test_indices]
    train_y, test_y = node_types[train_indices], node_types[test_indices]

    model.fit(train_x, train_y)

    test_accuracies.append(
      balanced_accuracy_score(test_y, model.predict(test_x))
    )

  mean_accuracy = np.mean(test_accuracies)
  std_accuracy = np.std(test_accuracies)
  
  return (mean_accuracy,std_accuracy)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import ShuffleSplit, StratifiedShuffleSplit
import numpy as np
from collections import Counter

def calculate_balanced_accuracy_from_vis_edge(visualizer, random_state=42,keep_top_classes=0):
  edges_embedded = visualizer._positive_edge_decomposition
  edge_types = visualizer._get_flatten_unknown_edge_types()
  # print(type(nodes_embedded))
  # print(nodes_embedded)
  # print(type(node_types))
  # print(node_types)
  return calculate_balanced_accuracy_edge(edges_embedded, edge_types, visualizer._graph, random_state,keep_top_classes)

def calculate_balanced_accuracy_edge(edges_embedded, edge_types, graph, random_state=42,keep_top_classes=0):
  # Sets the top k most frequent node types to be the only ones, the rest are set to k
  if(keep_top_classes>0):
    counts = np.bincount(edge_types)
    edge_type_names_iter = (
      graph.get_edge_type_name_from_edge_type_id(edge_id)
      for edge_id in range(graph.get_number_of_edge_types())
    )
    edge_type_names = np.array(
      list(edge_type_names_iter),
      dtype=str,
    )
    top_counts = [
      index
      for index, _ in sorted(
          enumerate(zip(counts, edge_type_names)), key=lambda x: x[1], reverse=True
      )[:keep_top_classes]
    ]
    #print(top_counts)
    for i, element_type in enumerate(edge_types):
      if element_type not in top_counts:
        edge_types[i] = keep_top_classes
      else:
        edge_types[i] = top_counts.index(element_type)

  if min(Counter(edge_types).values())==1:
    SplitterClass = ShuffleSplit
  else: 
    SplitterClass = StratifiedShuffleSplit

  test_accuracies = []

  for train_indices, test_indices in SplitterClass(
    n_splits=5,
    test_size=0.3,
    random_state=random_state
  ).split(edges_embedded, edge_types):
    
    model = DecisionTreeClassifier(max_depth=5)

    train_x, test_x = edges_embedded[train_indices], edges_embedded[test_indices]
    train_y, test_y = edge_types[train_indices], edge_types[test_indices]

    model.fit(train_x, train_y)

    test_accuracies.append(
      balanced_accuracy_score(test_y, model.predict(test_x))
    )

  mean_accuracy = np.mean(test_accuracies)
  std_accuracy = np.std(test_accuracies)
  
  return (mean_accuracy,std_accuracy)

In [ ]:
def print_accuracies_node(visualizers, names='', df=None, is_directed=False):
    names = [names for _ in range(len(visualizers))] if isinstance(names, str) else names
    for i,visualizer in enumerate(visualizers):
        (mean_1,std_1) = calculate_balanced_accuracy_from_vis(visualizer,keep_top_classes=7)
        (mean_2,std_2) = calculate_balanced_accuracy_from_vis(visualizer)
        print(f"Mean accuracy (7): {mean_1:.2%} ± {std_1:.2%}")
        print(f"Mean accuracy (all): {mean_2:.2%} ± {std_2:.2%}")
        num_types = num_node_types_directed if is_directed else num_node_types_undirected
        # check if df is not none
        if df is not None:
            df.loc[len(df.index)] = [names[i],is_directed,7,mean_1,std_1]
            df.loc[len(df.index)] = [names[i],is_directed,num_types,mean_2,std_2] 

def print_accuracies_edge(visualizers, names='', df=None, is_directed=False):
    names = [names for _ in range(len(visualizers))] if isinstance(names, str) else names
    for i,visualizer in enumerate(visualizers):
        (mean_1,std_1) = calculate_balanced_accuracy_from_vis_edge(visualizer,keep_top_classes=7)
        (mean_2,std_2) = calculate_balanced_accuracy_from_vis_edge(visualizer)
        print(f"Mean accuracy (7): {mean_1:.2%} ± {std_1:.2%}")
        print(f"Mean accuracy (all): {mean_2:.2%} ± {std_2:.2%}")
        num_types = num_edge_types_directed if is_directed else num_edge_types_undirected
        if df is not None:
            df.loc[len(df.index)] = [names[i],is_directed,7,mean_1,std_1]
            df.loc[len(df.index)] = [names[i],is_directed,num_types,mean_2,std_2] 

In [ ]:
def image_size_for_latex(points_width:float):
	inches_per_pt = 1.0/72.27               # Convert pt to inches
	golden_mean = (5**(1/2)-1.0)/2.0         # Aesthetic ratio
	fig_width = points_width*inches_per_pt  # width in inches
	fig_height = fig_width*golden_mean       # height in inches
	return (fig_width,fig_height)
fig_size_latex = image_size_for_latex(433.62*1.5) # thesis: 433.62pt, report: 345.0pt
fig_size_latex

## Line Undirected

### First order

In [ ]:
from grape.embedders import FirstOrderLINEEnsmallen
try:
    embedding_fo_line = cache.load_embeddings('FirstOrderLINEEnsmallen_100_fixed')
    print(type(embedding_fo_line))
except:
    start_time = time.time()
    embedding_fo_line_res = FirstOrderLINEEnsmallen(enable_cache=False)\
        .fit_transform(undirected_RNAKG)
    #cache.cache_embedding(embedding_fo_line_res, 'FirstOrderLINEEnsmallen_100_fixed')
    embedding_fo_line = cache.load_embeddings('FirstOrderLINEEnsmallen_100_fixed')
    end_time = time.time()
    print("FirstOrderLINEEnsmallen took: ", end_time-start_time)

#### Node embedding

In [ ]:
visualizers_fo_line = visualize.plot_embedding([embedding_fo_line], ["First-order LINE - Node types"], [undirected_RNAKG], 'node_types',
    save_path=save_folder+'node_type/FirstOrderLINEEnsmallen_100_fixed_node_types', 
    figsize=(fig_size_latex[1],fig_size_latex[1]), formats=['pdf','jpeg'], show_legend=True, show=False,
    k=7,
    number_of_subsampled_nodes=nodes_visualization_subsampling,
)

In [ ]:
print_accuracies_node(visualizers_fo_line,'First-Order LINE',df_node_type_pred,is_directed=False)

#### Edge embedding

In [ ]:
visualizers_fo_line_edge = visualize.plot_embedding([embedding_fo_line], ["First-order LINE - Edge types"], [undirected_RNAKG], 'edge_types',
    save_path=save_folder+'edge_type/FirstOrderLINEEnsmallen_100_fixed_edge_types', 
    figsize=(fig_size_latex[1],fig_size_latex[1]), formats=['pdf','jpeg'], show_legend=True, show=False,
    k=7,
    number_of_subsampled_edges=edges_visualization_subsampling,
)

In [ ]:
print_accuracies_edge(visualizers_fo_line_edge, 'First-Order LINE',df_edge_type_pred, is_directed=False)

### Second order

In [ ]:
from grape.embedders import SecondOrderLINEEnsmallen
try:
    embedding_so_line_undirected = cache.load_embeddings('SecondOrderLINEEnsmallen_100_fixed_undirected')
    print(type(embedding_so_line_undirected))
except:
    start_time = time.time()
    embedding_so_line_undirected_res = SecondOrderLINEEnsmallen(enable_cache=False)\
        .fit_transform(undirected_RNAKG)
    cache.cache_embedding(embedding_so_line_undirected_res, 'SecondOrderLINEEnsmallen_100_fixed_undirected')
    embedding_so_line_undirected = cache.load_embeddings('SecondOrderLINEEnsmallen_100_fixed_undirected')
    end_time = time.time()
    print("SecondOrderLINEEnsmallen took: ", end_time-start_time)

#### Node embedding

In [ ]:
visualizers_so_line = visualize.plot_embedding(
  [embedding_so_line_undirected[0], embedding_so_line_undirected[1],[embedding_so_line_undirected[0],embedding_so_line_undirected[1]]], 
  ["Second-order LINE (undirected) (0)","Second-order LINE (undirected) (1)","Second-order LINE (undirected) (0+1)"], 
  [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
  save_path=save_folder+'node_type/SOLINE_undirected_comparison', 
  figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
  show=False,
  visualization_type='node_types',
  show_legend=True,
  k=7
)

In [ ]:
print_accuracies_node(visualizers_so_line,['Second-Order LINE (0)','Second-Order LINE (1)','Second-Order LINE (0+1)'],df_node_type_pred,is_directed=False)

#### Edge embedding

In [ ]:
visualizers_so_line_edge = visualize.plot_embedding(
  [embedding_so_line_undirected[0], embedding_so_line_undirected[1],[embedding_so_line_undirected[0],embedding_so_line_undirected[1]]], 
  ["Second-order LINE (undirected) (0)","Second-order LINE (undirected) (1)","Second-order LINE (undirected) (0+1)"], 
  [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
  save_path=save_folder+'edge_type/SOLINE_undirected_comparison_edge', 
  figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
  show=False,
  visualization_type='edge_types',
  show_legend=True,
  k=7
)

In [ ]:
print_accuracies_edge(visualizers_so_line_edge,['Second-Order LINE (0)','Second-Order LINE (1)','Second-Order LINE (0+1)'],df_edge_type_pred,is_directed=False)

## Line Directed

### Second Order LINE

In [ ]:
from grape.embedders import SecondOrderLINEEnsmallen
try:
    embedding_so_line_directed = cache.load_embeddings('SecondOrderLINEEnsmallen_100_fixed_directed')
    print(type(embedding_so_line_directed))
except:
    start_time = time.time()
    embedding_so_line_directed_res = SecondOrderLINEEnsmallen(enable_cache=False)\
        .fit_transform(directed_RNAKG)
    cache.cache_embedding(embedding_so_line_directed_res, 'SecondOrderLINEEnsmallen_100_fixed_directed')
    embedding_so_line_directed = cache.load_embeddings('SecondOrderLINEEnsmallen_100_fixed_directed')
    end_time = time.time()
    print("SecondOrderLINEEnsmallen took: ", end_time-start_time)

#### Node embedding

In [ ]:
visualizers_so_line_directed = visualize.plot_embedding(
  [embedding_so_line_directed[0], embedding_so_line_directed[1],[embedding_so_line_directed[0],embedding_so_line_directed[1]]], 
  ["Second-order LINE (directed) (0)","Second-order LINE (directed) (1)","Second-order LINE (directed) (0+1)"], 
  [directed_RNAKG,directed_RNAKG,directed_RNAKG],
  save_path=save_folder+'node_type/SOLINE_directed_comparison', 
  figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
  show=False,
  visualization_type='node_types',
  show_legend=True,
  k=7
)

In [ ]:
print_accuracies_node(visualizers_so_line_directed,['Second-Order LINE (0)','Second-Order LINE (1)','Second-Order LINE (0+1)'],df_node_type_pred,is_directed=True)

#### Edge embedding

In [ ]:
visualizers_so_line_directed_edge = visualize.plot_embedding(
  [embedding_so_line_directed[0], embedding_so_line_directed[1],[embedding_so_line_directed[0],embedding_so_line_directed[1]]], 
  ["Second-order LINE (directed) (0)","Second-order LINE (directed) (1)","Second-order LINE (directed) (0+1)"], 
  [directed_RNAKG,directed_RNAKG,directed_RNAKG],
  save_path=save_folder+'edge_type/SOLINE_directed_comparison_edge', 
  figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
  show=False,
  visualization_type='edge_types',
  show_legend=True,
  k=7
)

In [ ]:
print_accuracies_edge(visualizers_so_line_directed_edge,['Second-Order LINE (0)','Second-Order LINE (1)','Second-Order LINE (0+1)'],df_edge_type_pred,is_directed=True)

## LINE Comparison

### First vs Second Order undirected

In [ ]:
visualizers = visualize.plot_embedding(
    [embedding_fo_line[0], [embedding_so_line_undirected[0],embedding_so_line_undirected[1]]], 
    ["First-order LINE", "Second-order LINE"], 
    [undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'node_type/LINE_Comparison', 
    figsize=(fig_size_latex[1]*2,fig_size_latex[1]),
    show=False,
    visualization_type='node_types',
    show_legend=True,
    k=7
)

In [ ]:
visualizers = visualize.plot_embedding(
    [embedding_fo_line[0], [embedding_so_line_undirected[0],embedding_so_line_undirected[1]]], 
    ["", ""], 
    [undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'node_type/LINE_Comparison_no_title', 
    figsize=(fig_size_latex[1]*2,fig_size_latex[1]),
    show=False,
    visualization_type='node_types',
    show_legend=True,
    k=7
)

In [ ]:
visualizers = visualize.plot_embedding(
    [embedding_fo_line[0], [embedding_so_line_undirected[0],embedding_so_line_undirected[1]]], 
    ["First-order LINE", "Second-order LINE"], 
    [undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'edge_type/LINE_Comparison_edge', 
    figsize=(fig_size_latex[1]*2,fig_size_latex[1]),
    show=False,
    visualization_type='edge_types',
    show_legend=True,
    k=7
)

### Second order Undirected vs Directed

In [ ]:
visualizers = visualize.plot_embedding(
    [[embedding_so_line_undirected[0],embedding_so_line_undirected[1]], embedding_so_line_directed[0]], 
    ["Second-order LINE (undirected)", "Second-order LINE (directed)"], 
    [undirected_RNAKG,directed_RNAKG],
    save_path=save_folder+'node_type/SOLine_directed_undirected_comparison', 
    figsize=(fig_size_latex[1]*2,fig_size_latex[1]),
    show=False,
    visualization_type='node_types',
    show_legend=True,
    k=7
)

In [ ]:
visualizers = visualize.plot_embedding(
    [[embedding_so_line_undirected[0],embedding_so_line_undirected[1]], embedding_so_line_directed[0]], 
    ["Second-order LINE (undirected)", "Second-order LINE (directed)"], 
    [undirected_RNAKG,directed_RNAKG],
    save_path=save_folder+'edge_type/SOLine_directed_undirected_comparison_edge', 
    figsize=(fig_size_latex[1]*2,fig_size_latex[1]),
    show=False,
    visualization_type='edge_types',
    show_legend=True,
    k=7
)

## Node2Vec Skipgram Embedding
### DFS

In [ ]:
from grape.embedders import Node2VecSkipGramEnsmallen
try:
    embedding_n2v_sg_dfs = cache.load_embeddings('Node2VecSkipGramEnsmallen_DFS_100_fixed2')
    print(type(embedding_n2v_sg_dfs))
except:
    start_time = time.time()
    embedding_n2v_sg_dfs_res = Node2VecSkipGramEnsmallen(enable_cache=False,return_weight=0.2, explore_weight=5)\
        .fit_transform(undirected_RNAKG)
    cache.cache_embedding(embedding_n2v_sg_dfs_res, 'Node2VecSkipGramEnsmallen_DFS_100_fixed2')
    embedding_n2v_sg_dfs = cache.load_embeddings('Node2VecSkipGramEnsmallen_DFS_100_fixed2')
    end_time = time.time()
    print("Node2VecSkipGramEnsmallen DFS took: ", end_time-start_time)

#### Node embedding

In [ ]:
visualizers_n2vsg_dfs = visualize.plot_embedding(
    [embedding_n2v_sg_dfs[0], embedding_n2v_sg_dfs[1],[embedding_n2v_sg_dfs[0], embedding_n2v_sg_dfs[1]]], 
    ["DFS - Central tokens based", "DFS - Contextual tokens based", "DFS - Both"], 
    [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'node_type/N2VSG_DFS_comparison', 
    figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
    show=False,
    visualization_type='node_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_node(visualizers_n2vsg_dfs,['Node2Vec SkipGram DFS (0)', 'Node2Vec SkipGram DFS (1)','Node2Vec SkipGram DFS (0+1)'],df_node_type_pred)

#### Edge embedding

In [ ]:
visualizers_n2vsg_dfs_edge = visualize.plot_embedding(
    [embedding_n2v_sg_dfs[0], embedding_n2v_sg_dfs[1],[embedding_n2v_sg_dfs[0], embedding_n2v_sg_dfs[1]]], 
    ["DFS - Central tokens based", "DFS - Contextual tokens based", "DFS - Both"], 
    [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'edge_type/N2VSG_DFS_comparison_edge', 
    figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
    show=False,
    visualization_type='edge_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_edge(visualizers_n2vsg_dfs_edge,['Node2Vec SkipGram DFS (0)', 'Node2Vec SkipGram DFS (1)','Node2Vec SkipGram DFS (0+1)'],df_edge_type_pred)

### BFS

In [ ]:
from grape.embedders import Node2VecSkipGramEnsmallen
try:
    embedding_n2v_sg_bfs = cache.load_embeddings('Node2VecSkipGramEnsmallen_BFS_100_fixed2')
    print(type(embedding_n2v_sg_bfs))
except:
    start_time = time.time()
    embedding_n2v_sg_bfs_res = Node2VecSkipGramEnsmallen(enable_cache=False, return_weight=5, explore_weight=0.2)\
        .fit_transform(undirected_RNAKG)
    cache.cache_embedding(embedding_n2v_sg_bfs_res, 'Node2VecSkipGramEnsmallen_BFS_100_fixed2')
    embedding_n2v_sg_bfs = cache.load_embeddings('Node2VecSkipGramEnsmallen_BFS_100_fixed2')
    end_time = time.time()
    print("Node2VecSkipGramEnsmallen BFS took: ", end_time-start_time)

#### Node embedding

In [ ]:
visualizers_n2vsg_bfs = visualize.plot_embedding(
    [embedding_n2v_sg_bfs[0], embedding_n2v_sg_bfs[1],[embedding_n2v_sg_bfs[0], embedding_n2v_sg_bfs[1]]], 
    ["BFS - Central tokens based", "BFS - Contextual tokens based", "BFS - Both"], 
    [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'node_type/N2VSG_BFS_comparison', 
    figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
    show=False,
    visualization_type='node_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_node(visualizers_n2vsg_bfs,['Node2Vec SkipGram BFS (0)', 'Node2Vec SkipGram BFS (1)','Node2Vec SkipGram BFS (0+1)'],df_node_type_pred)

#### Edge embedding

In [ ]:
visualizers_n2vsg_bfs_edge = visualize.plot_embedding(
    [embedding_n2v_sg_bfs[0], embedding_n2v_sg_bfs[1],[embedding_n2v_sg_bfs[0], embedding_n2v_sg_bfs[1]]], 
    ["BFS - Central tokens based", "BFS - Contextual tokens based", "BFS - Both"], 
    [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'edge_type/N2VSG_BFS_comparison_edge', 
    figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
    show=False,
    visualization_type='edge_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_edge(visualizers_n2vsg_bfs_edge,['Node2Vec SkipGram BFS (0)', 'Node2Vec SkipGram BFS (1)','Node2Vec SkipGram BFS (0+1)'],df_edge_type_pred)

### Balanced

In [ ]:
from grape.embedders import Node2VecSkipGramEnsmallen
try:
    embedding_n2v_sg_balanced = cache.load_embeddings('Node2VecSkipGramEnsmallen_Balanced_100_fixed2')
    print(type(embedding_n2v_sg_balanced))
except:
    start_time = time.time()
    embedding_n2v_sg_balanced_res = Node2VecSkipGramEnsmallen(enable_cache=False, return_weight=1, explore_weight=1)\
        .fit_transform(undirected_RNAKG)
    cache.cache_embedding(embedding_n2v_sg_balanced_res, 'Node2VecSkipGramEnsmallen_Balanced_100_fixed2')
    embedding_n2v_sg_balanced = cache.load_embeddings('Node2VecSkipGramEnsmallen_Balanced_100_fixed2')
    end_time = time.time()
    print("Node2VecSkipGramEnsmallen Balanced took: ", end_time-start_time)

#### Node embedding

In [ ]:
visualizers_n2vsg_balanced = visualize.plot_embedding(
    [embedding_n2v_sg_balanced[0], embedding_n2v_sg_balanced[1],[embedding_n2v_sg_balanced[0], embedding_n2v_sg_balanced[1]]], 
    ["Balanced - Central tokens based", "Balanced - Contextual tokens based", "Balanced - Both"], 
    [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'node_type/N2VSG_Balanced_comparison', 
    figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
    show=False,
    visualization_type='node_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_node(visualizers_n2vsg_balanced,['Node2Vec SkipGram Balanced (0)', 'Node2Vec SkipGram Balanced (1)','Node2Vec SkipGram Balanced (0+1)'],df_node_type_pred)

#### Edge embedding

In [ ]:
visualizers_n2vsg_balanced_edge = visualize.plot_embedding(
    [embedding_n2v_sg_balanced[0], embedding_n2v_sg_balanced[1],[embedding_n2v_sg_balanced[0], embedding_n2v_sg_balanced[1]]], 
    ["Balanced - Central tokens based", "Balanced - Contextual tokens based", "Balanced - Both"], 
    [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'edge_type/N2VSG_Balanced_comparison_edge', 
    figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
    show=False,
    visualization_type='edge_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_edge(visualizers_n2vsg_balanced_edge,['Node2Vec SkipGram Balanced (0)', 'Node2Vec SkipGram Balanced (1)','Node2Vec SkipGram Balanced (0+1)'],df_edge_type_pred)

### Comparison

#### DFS vs BFS vs Balanced

In [ ]:
visualizers_n2v_sg_comp = visualize.plot_embedding(
    [embedding_n2v_sg_dfs[0], embedding_n2v_sg_bfs[0],embedding_n2v_sg_balanced[0]], 
    ["DFS", "BFS", "Balanced"], 
    [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'node_type/N2VSG_comparison_parameters', 
    figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
    show=False,
    visualization_type='node_types',
    show_legend=True,
    k=7
)

In [ ]:
visualizers_n2v_sg_comp = visualize.plot_embedding(
    [embedding_n2v_sg_dfs[0], embedding_n2v_sg_bfs[0],embedding_n2v_sg_balanced[0]], 
    ["", "", ""], 
    [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'node_type/N2VSG_comparison_parameters_no_title', 
    figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
    show=False,
    visualization_type='node_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_node(visualizers_n2v_sg_comp)

In [ ]:
visualizers_n2v_sg_comp_edge = visualize.plot_embedding(
    [embedding_n2v_sg_dfs[0], embedding_n2v_sg_bfs[0],embedding_n2v_sg_balanced[0]], 
    ["DFS", "BFS", "Balanced"], 
    [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'edge_type/N2VSG_comparison_parameters_edge', 
    figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
    show=False,
    visualization_type='edge_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_edge(visualizers_n2v_sg_comp_edge)

## Comparison: Node2Vec Skipgram and LINE

### Node types

In [ ]:
visualizers_line_n2vsg_comp = visualize.plot_embedding(
    [embedding_fo_line, embedding_n2v_sg_dfs[0]], 
    ["FO-LINE", "N2V-SG-DFS"], 
    [undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'node_type/LINE_N2VSG_comparison', 
    figsize=(fig_size_latex[1]*2,fig_size_latex[1]),
    show=False,
    visualization_type='node_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_node(visualizers_line_n2vsg_comp)

### Edge types

In [ ]:
visualizers_line_n2vsg_comp_edge = visualize.plot_embedding(
    [[embedding_so_line_directed[0],embedding_so_line_directed[1]], embedding_n2v_sg_balanced[0]], 
    ["SO-LINE-Directed-Comb", "N2V-SG-Balanced"], 
    [directed_RNAKG,undirected_RNAKG],
    save_path=save_folder+'edge_type/LINE_N2VSG_comparison_edge', 
    figsize=(fig_size_latex[1]*2,fig_size_latex[1]),
    show=False,
    visualization_type='edge_types',
    show_legend=True,
    k=7
)

In [ ]:
visualize.plot_embedding(
    [[embedding_so_line_directed[0],embedding_so_line_directed[1]], embedding_n2v_sg_balanced[0]], 
    ["", ""], 
    [directed_RNAKG,undirected_RNAKG],
    save_path=save_folder+'edge_type/LINE_N2VSG_comparison_edge_no_title', 
    figsize=(fig_size_latex[1]*2,fig_size_latex[1]),
    show=False,
    visualization_type='edge_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_edge(visualizers_line_n2vsg_comp_edge)

## Node2Vec CBOW Embedding
### DFS

In [ ]:
from grape.embedders import Node2VecCBOWEnsmallen
try:
    embedding_n2v_cbow_dfs = cache.load_embeddings('Node2VecCBOWEnsmallen_DFS_100_fixed2')
    print(type(embedding_n2v_cbow_dfs))
except:
    start_time = time.time()
    embedding_n2v_cbow_dfs_res = Node2VecCBOWEnsmallen(enable_cache=False, return_weight=0.2, explore_weight=5)\
        .fit_transform(undirected_RNAKG)
    cache.cache_embedding(embedding_n2v_cbow_dfs_res, 'Node2VecCBOWEnsmallen_DFS_100_fixed2')
    embedding_n2v_cbow_dfs = cache.load_embeddings('Node2VecCBOWEnsmallen_DFS_100_fixed2')
    end_time = time.time()
    print("Node2VecCBOWEnsmallen DFS took: ", end_time-start_time)

#### Node embedding

In [ ]:
visualizers_n2vcbow_dfs = visualize.plot_embedding(
    [embedding_n2v_cbow_dfs[0], embedding_n2v_cbow_dfs[1],[embedding_n2v_cbow_dfs[0], embedding_n2v_cbow_dfs[1]]], 
    ["DFS - Central tokens based", "DFS - Contextual tokens based", "DFS - Both"], 
    [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'node_type/N2VCBOW_DFS_comparison', 
    figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
    show=False,
    visualization_type='node_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_node(visualizers_n2vcbow_dfs,['Node2Vec CBOW DFS (0)', 'Node2Vec CBOW DFS (1)','Node2Vec CBOW DFS (0+1)'],df_node_type_pred)

#### Edge embedding

In [ ]:
visualizers_n2vcbow_dfs_edge = visualize.plot_embedding(
    [embedding_n2v_cbow_dfs[0], embedding_n2v_cbow_dfs[1],[embedding_n2v_cbow_dfs[0], embedding_n2v_cbow_dfs[1]]], 
    ["DFS - Central tokens based", "DFS - Contextual tokens based", "DFS - Both"], 
    [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'edge_type/N2VCBOW_DFS_comparison_edge', 
    figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
    show=False,
    visualization_type='edge_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_edge(visualizers_n2vcbow_dfs_edge,['Node2Vec CBOW DFS (0)', 'Node2Vec CBOW DFS (1)','Node2Vec CBOW DFS (0+1)'],df_edge_type_pred)

### BFS

In [ ]:
from grape.embedders import Node2VecCBOWEnsmallen
try:
    embedding_n2v_cbow_bfs = cache.load_embeddings('Node2VecCBOWEnsmallen_BFS_100_fixed2')
    print(type(embedding_n2v_cbow_bfs))
except:
    start_time = time.time()
    embedding_n2v_cbow_bfs_res = Node2VecCBOWEnsmallen(enable_cache=False, return_weight=5, explore_weight=0.2)\
        .fit_transform(undirected_RNAKG)
    cache.cache_embedding(embedding_n2v_cbow_bfs_res, 'Node2VecCBOWEnsmallen_BFS_100_fixed2')
    embedding_n2v_cbow_bfs = cache.load_embeddings('Node2VecCBOWEnsmallen_BFS_100_fixed2')
    end_time = time.time()
    print("Node2VecCBOWEnsmallen BFS took: ", end_time-start_time)

#### Node embedding

In [ ]:
visualizers_n2vcbow_bfs = visualize.plot_embedding(
    [embedding_n2v_cbow_bfs[0], embedding_n2v_cbow_bfs[1],[embedding_n2v_cbow_bfs[0], embedding_n2v_cbow_bfs[1]]], 
    ["BFS - Central tokens based", "BFS - Contextual tokens based", "BFS - Both"], 
    [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'node_type/N2VCBOW_BFS_comparison', 
    figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
    show=False,
    visualization_type='node_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_node(visualizers_n2vcbow_bfs,['Node2Vec CBOW BFS (0)', 'Node2Vec CBOW BFS (1)','Node2Vec CBOW BFS (0+1)'],df_node_type_pred)

#### Edge embedding

In [ ]:
visualizers_n2vcbow_bfs_edge = visualize.plot_embedding(
    [embedding_n2v_cbow_bfs[0], embedding_n2v_cbow_bfs[1],[embedding_n2v_cbow_bfs[0], embedding_n2v_cbow_bfs[1]]], 
    ["BFS - Central tokens based", "BFS - Contextual tokens based", "BFS - Both"], 
    [undirected_RNAKG,undirected_RNAKG,undirected_RNAKG],
    save_path=save_folder+'edge_type/N2VCBOW_BFS_comparison_edge', 
    figsize=(fig_size_latex[1]*3,fig_size_latex[1]),
    show=False,
    visualization_type='edge_types',
    show_legend=True,
    k=7
)

In [ ]:
print_accuracies_edge(visualizers_n2vcbow_bfs_edge,['Node2Vec CBOW BFS (0)', 'Node2Vec CBOW BFS (1)','Node2Vec CBOW BFS (0+1)'],df_edge_type_pred)

## Dataframes with balanced accuracies

In [ ]:
# df_node_type_pred.to_csv('node_type_pred_results.csv')
# df_edge_type_pred.to_csv('edge_type_pred_results.csv')
df_node_type_pred_copy = df_node_type_pred.copy()

In [ ]:
df_node_type_pred['balanced_acc_mean_string'] = df_node_type_pred['balanced_acc_mean'].apply(lambda x: f'{x:.2%}')
df_node_type_pred['balanced_acc_std_string'] = df_node_type_pred['balanced_acc_std'].apply(lambda x: f'{x:.2%}')
df_node_type_pred['balanced_acc_comb'] = df_node_type_pred['balanced_acc_mean_string'] + ' ± ' + df_node_type_pred['balanced_acc_std_string']
df_node_type_pred['balanced_acc_comb'] = df_node_type_pred['balanced_acc_comb'].apply(lambda x:x.replace("%","\%"))

In [ ]:
df_edge_type_pred['balanced_acc_mean_string'] = df_edge_type_pred['balanced_acc_mean'].apply(lambda x: f'{x:.2%}')
df_edge_type_pred['balanced_acc_std_string'] = df_edge_type_pred['balanced_acc_std'].apply(lambda x: f'{x:.2%}')
df_edge_type_pred['balanced_acc_comb'] = df_edge_type_pred['balanced_acc_mean_string'] + ' ± ' + df_edge_type_pred['balanced_acc_std_string']
df_edge_type_pred['balanced_acc_comb'] = df_edge_type_pred['balanced_acc_comb'].apply(lambda x:x.replace("%","\%"))

In [ ]:
df_node_type_pred[df_node_type_pred['embedding_name'].str.contains('Second-Order')].sort_values(by='balanced_acc_mean', ascending=False)

In [ ]:
df_node_type_pred[df_node_type_pred['embedding_name'].str.contains('Order')].sort_values(by='balanced_acc_mean', ascending=False)

In [ ]:
df_edge_type_pred[df_edge_type_pred['embedding_name'].str.contains('Second-Order')].sort_values(by='balanced_acc_mean', ascending=False)

In [ ]:
df_edge_type_pred[df_edge_type_pred['embedding_name'].str.contains('Order')].sort_values(by='balanced_acc_mean', ascending=False)

In [ ]:
df_node_type_pred[df_node_type_pred['embedding_name'].str.contains('Node2Vec')][df_node_type_pred['embedding_name'].str.contains('DFS')][df_node_type_pred['num_classes']==7]

In [ ]:
df_node_type_pred[df_node_type_pred['embedding_name'].str.contains('Node2Vec')][df_node_type_pred['embedding_name'].str.contains('BFS')][df_node_type_pred['num_classes']==7]

In [ ]:
df_node_type_pred[df_node_type_pred['embedding_name'].str.contains('Node2Vec SkipGram')][df_node_type_pred['embedding_name'].str.endswith('(0)')]

In [ ]:
df_edge_type_pred[df_edge_type_pred['embedding_name'].str.contains('Node2Vec SkipGram')][df_edge_type_pred['embedding_name'].str.endswith('(0)')]

In [ ]:
df_node_type_pred[df_node_type_pred['embedding_name'].str.contains('Node2Vec SkipGram')].sort_values('balanced_acc_mean',ascending=False)